In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
ICE-SHEET LEVEL-2 ICL DATASET GENERATOR
=======================================

Canonical trajectory-based operator design
------------------------------------------
Each physical operator/system is defined by one fixed pair

    operator_j = (A_j, beta_j(x,y))

where
    A_j = flow_A_base_per_year * A_rel_j,
    A_rel_j in [0.5, 2.0],

and beta_j(x,y) is one independent GP basal-friction realization.

For each operator j:
    * A_j is fixed for the entire trajectory.
    * beta_j is fixed for the entire trajectory.
    * LF and HF use the same underlying (A_j, beta_j), but different grids.
    * We evolve one trajectory from t=0 to t=50.
    * We save complete snapshots at t = 1,2,...,50.

Dataset:
    100 operators x 50 times
    70 train systems + 30 completely held-out ID systems
    LF grid = 15 x 15
    HF grid = 41 x 41

Intended ICL episode:
    demo_i = [XY, t_i, Uc(t_i), Uf(t_i)]
    query  = [XY, t_q, Uc(t_q)]
    target = Uf(t_q)

A, beta, and H are stored for physics/audit but are NOT intended as model inputs.

Episode-design rule:
    This dataset does NOT predefine demonstration times or query times.
    Time is only the within-operator sample axis t = 1,...,50.
    Training/evaluation code constructs ICL episodes dynamically.
    A fixed RNG seed may be used in evaluation for reproducibility.

Scope:
    This version uses a held-out in-distribution system split only.
    No temporal OOD split is built into the dataset.

Resume / sharding:
    OP_START=0 OP_STOP=10 python this_script.py
    ...
    ASSEMBLE_ONLY=1 python this_script.py

Only output path/sharding controls are configurable by environment variables;
the scientific design below is intentionally locked.
"""

from __future__ import annotations

import hashlib
import json
import math
import os
import time
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Dict, Tuple

import numpy as np
from scipy.interpolate import RegularGridInterpolator
from scipy.sparse import coo_matrix, csc_matrix
from scipy.sparse.linalg import splu


# =============================================================================
# CONFIGURATION
# =============================================================================

@dataclass
class Config:
    # Reproducibility.
    beta_seed: int = 20260901
    operator_split_seed: int = 20260902

    # Canonical Level-2 system/operator design.
    n_operators: int = 100
    n_train_operators: int = 70
    n_saved_times: int = 50
    A_rel_min: float = 0.5
    A_rel_max: float = 2.0

    # Howard-style random basal-friction setup.
    domain_m: float = 60_000.0
    beta_corr_m: float = 54_000.0
    beta_cov_amplitude: float = 1.0
    beta_gp_scale: float = 0.35
    beta_opt_base: float = 3000.0  # Pa yr / m

    # Multiresolution grids.
    n_lf: int = 15
    n_hf: int = 41

    # Trajectory axis.
    final_year: int = 50
    evolution_step_years: float = 1.0

    # Physical constants / reproduction choices.
    rho_ice: float = 910.0
    gravity: float = 9.8101
    glen_n: float = 3.0
    flow_A_base_per_year: float = 1.0e-16  # Pa^-3 yr^-1

    # Halfar-style initial dome.
    halfar_H0_m: float = 2000.0 * math.sqrt(0.125)
    halfar_R0_m: float = 60_000.0 * math.sqrt(0.125)

    # Reduced nonlinear membrane-style stress balance.
    strain_reg_per_year: float = 1.0e-5
    viscosity_min_Pa_yr: float = 1.0e5
    viscosity_max_Pa_yr: float = 1.0e10
    max_picard_iters: int = 30
    picard_tol: float = 1.0e-5
    picard_relax: float = 0.65
    min_H_for_stress_m: float = 1.0

    # Neutral/no-calibration values.
    membrane_scale: float = 1.0
    friction_response_power: float = 1.0

    # Thickness transport.
    cfl: float = 0.45
    max_transport_substep_yr: float = 0.20
    thickness_clip_max_m: float = 5000.0


    # Output.
    out_dir: str = "ice_icl_Abeta100_time1to50_fullfield_noFixedTimeSplit"
    master_name: str = "ice_icl_Abeta100_time1to50_fullfield_noFixedTimeSplit_master.npz"
    dtype: str = "float32"


MAX_PICARD_BLOCKS = 5
PICARD_RELAX_CANDIDATES = (0.65, 0.50, 0.35, 0.25)

OVERWRITE_RUNS = os.environ.get("OVERWRITE_RUNS", "0").strip() == "1"
ASSEMBLE_ONLY = os.environ.get("ASSEMBLE_ONLY", "0").strip() == "1"
OUT_DIR_OVERRIDE = os.environ.get("OUT_DIR", "").strip() or None
OP_START_ENV = os.environ.get("OP_START", "").strip()
OP_STOP_ENV = os.environ.get("OP_STOP", "").strip()


def build_config() -> Config:
    cfg = Config()
    if OUT_DIR_OVERRIDE is not None:
        cfg.out_dir = str(OUT_DIR_OVERRIDE)
    return cfg


# =============================================================================
# GRID / FIELD HELPERS
# =============================================================================


def grid_xy(n: int, L: float) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    x = np.linspace(0.0, L, n, dtype=np.float64)
    y = np.linspace(0.0, L, n, dtype=np.float64)
    X, Y = np.meshgrid(x, y, indexing="xy")
    return x, y, X, Y


def beta_opt_field(X: np.ndarray, Y: np.ndarray, cfg: Config) -> np.ndarray:
    return cfg.beta_opt_base * (
        1.0
        + 0.5
        * np.sin(np.pi * X / 30_000.0)
        * np.sin(np.pi * Y / 30_000.0)
    )


def halfar_initial_thickness(X: np.ndarray, Y: np.ndarray, cfg: Config) -> np.ndarray:
    """Smooth radially symmetric Halfar-style initial dome."""
    xc = 0.5 * cfg.domain_m
    yc = 0.5 * cfg.domain_m
    r = np.sqrt((X - xc) ** 2 + (Y - yc) ** 2)

    n = cfg.glen_n
    q = np.maximum(
        0.0,
        1.0 - (r / cfg.halfar_R0_m) ** ((n + 1.0) / n),
    )
    H = cfg.halfar_H0_m * q ** (n / (2.0 * n + 1.0))
    return H.astype(np.float64)


def bilinear_resample(
    field: np.ndarray,
    x_src: np.ndarray,
    y_src: np.ndarray,
    x_dst: np.ndarray,
    y_dst: np.ndarray,
) -> np.ndarray:
    interp = RegularGridInterpolator(
        (y_src, x_src),
        field,
        method="linear",
        bounds_error=True,
    )
    Xd, Yd = np.meshgrid(x_dst, y_dst, indexing="xy")
    pts = np.column_stack([Yd.ravel(), Xd.ravel()])
    return interp(pts).reshape(len(y_dst), len(x_dst))


def grad2(f: np.ndarray, dx: float) -> Tuple[np.ndarray, np.ndarray]:
    fy, fx = np.gradient(f, dx, dx, edge_order=2)
    return fx, fy


def rel_l2(a: np.ndarray, b: np.ndarray, eps: float = 1.0e-12) -> float:
    a = np.asarray(a, dtype=np.float64)
    b = np.asarray(b, dtype=np.float64)
    return float(
        np.linalg.norm(a.ravel() - b.ravel())
        / (np.linalg.norm(b.ravel()) + eps)
    )





# =============================================================================
# OPERATOR BANK: ONE (A, beta) PAIR PER OPERATOR
# =============================================================================

def build_or_load_operator_bank(cfg: Config, out_dir: Path) -> Dict[str, np.ndarray]:
    """
    Build 100 physical systems/operators.

    operator_j = (A_j, beta_j(x,y))

    A_rel values are evenly spaced over [0.5, 2.0].
    Each operator receives its OWN independent beta GP realization.
    The 70/30 split is performed at the operator/system level.
    """
    path = out_dir / "operator_bank_Abeta100_split70_30_gpScale0p35.npz"

    if path.exists():
        z = np.load(path, allow_pickle=False)
        valid = (
            int(z["n_operators"]) == cfg.n_operators
            and int(z["n_train_operators"]) == cfg.n_train_operators
            and int(z["beta_seed"]) == cfg.beta_seed
            and int(z["split_seed"]) == cfg.operator_split_seed
            and abs(float(z["beta_gp_scale"]) - cfg.beta_gp_scale) < 1.0e-12
            and abs(float(z["A_rel_min"]) - cfg.A_rel_min) < 1.0e-12
            and abs(float(z["A_rel_max"]) - cfg.A_rel_max) < 1.0e-12
        )
        if valid:
            print(f"[operators] Loading cached operator bank: {path}")
            return {k: z[k] for k in z.files}

    # A parameter for each physical system.
    A_rel_values = np.linspace(
        cfg.A_rel_min,
        cfg.A_rel_max,
        cfg.n_operators,
        dtype=np.float64,
    )
    A_values = cfg.flow_A_base_per_year * A_rel_values

    # Completely held-out in-distribution SYSTEM split.
    rng_split = np.random.default_rng(cfg.operator_split_seed)
    perm = rng_split.permutation(cfg.n_operators)
    train_operator_indices = np.sort(
        perm[:cfg.n_train_operators]
    ).astype(np.int64)
    test_operator_indices = np.sort(
        perm[cfg.n_train_operators:]
    ).astype(np.int64)

    split_labels = np.full(cfg.n_operators, -1, dtype=np.int8)
    split_labels[train_operator_indices] = 0
    split_labels[test_operator_indices] = 1

    # One independent beta GP field per operator.
    xh, yh, Xh, Yh = grid_xy(cfg.n_hf, cfg.domain_m)
    xl, yl, _, _ = grid_xy(cfg.n_lf, cfg.domain_m)

    pts = np.column_stack([Xh.ravel(), Yh.ravel()])
    sq = np.sum(pts * pts, axis=1)
    d2 = sq[:, None] + sq[None, :] - 2.0 * (pts @ pts.T)
    d2 = np.maximum(d2, 0.0)

    K = cfg.beta_cov_amplitude * np.exp(
        -0.5 * d2 / (cfg.beta_corr_m ** 2)
    )
    K.flat[::K.shape[0] + 1] += 1.0e-10

    print(f"[beta] independent operator beta fields; GP scale s={cfg.beta_gp_scale}")
    print(f"[beta] Cholesky factorization of {K.shape[0]} x {K.shape[0]} covariance ...")
    Lc = np.linalg.cholesky(K)

    mean_gamma = np.log(beta_opt_field(Xh, Yh, cfg).ravel())
    rng_beta = np.random.default_rng(cfg.beta_seed)
    Z = rng_beta.standard_normal((K.shape[0], cfg.n_operators))
    gamma = mean_gamma[:, None] + cfg.beta_gp_scale * (Lc @ Z)

    beta_hf = np.exp(gamma.T).reshape(
        cfg.n_operators,
        cfg.n_hf,
        cfg.n_hf,
    )

    beta_lf = np.empty(
        (cfg.n_operators, cfg.n_lf, cfg.n_lf),
        dtype=np.float64,
    )
    for op_idx in range(cfg.n_operators):
        beta_lf[op_idx] = bilinear_resample(
            beta_hf[op_idx], xh, yh, xl, yl
        )

    payload = dict(
        A_rel_values=A_rel_values.astype(cfg.dtype),
        A_values=A_values.astype(np.float64),
        beta_hf=beta_hf.astype(cfg.dtype),
        beta_lf=beta_lf.astype(cfg.dtype),
        train_operator_indices=train_operator_indices,
        test_operator_indices=test_operator_indices,
        operator_split_labels=split_labels,
        x_hf=xh.astype(cfg.dtype),
        y_hf=yh.astype(cfg.dtype),
        x_lf=xl.astype(cfg.dtype),
        y_lf=yl.astype(cfg.dtype),
        n_operators=np.array(cfg.n_operators, dtype=np.int64),
        n_train_operators=np.array(cfg.n_train_operators, dtype=np.int64),
        n_test_operators=np.array(
            cfg.n_operators - cfg.n_train_operators,
            dtype=np.int64,
        ),
        beta_seed=np.array(cfg.beta_seed, dtype=np.int64),
        split_seed=np.array(cfg.operator_split_seed, dtype=np.int64),
        beta_gp_scale=np.array(cfg.beta_gp_scale, dtype=np.float64),
        A_rel_min=np.array(cfg.A_rel_min, dtype=np.float64),
        A_rel_max=np.array(cfg.A_rel_max, dtype=np.float64),
        flow_A_base_per_year=np.array(
            cfg.flow_A_base_per_year, dtype=np.float64
        ),
    )

    np.savez_compressed(path, **payload)
    print(f"[operators] Saved: {path}")
    print("[operators] train indices:", train_operator_indices.tolist())
    print("[operators] test  indices:", test_operator_indices.tolist())
    return payload


# =============================================================================
# REDUCED MOLHO-LIKE / MEMBRANE-STYLE SOLVER
# =============================================================================


class ReducedIceFlowSolver:
    """
    Nonlinear 2-D membrane-style stress-balance approximation.

    Componentwise system:
        -div(K grad u) + beta u = -rho g H dH/dx
        -div(K grad v) + beta v = -rho g H dH/dy

    where
        K = 2 mu H
        mu = 1/2 A^(-1/n) De^(1/n - 1)

    NEW operator variation:
        A = flow_A_base_per_year * A_rel

    The same equations are used at 15x15 and 41x41; only resolution changes.
    """

    def __init__(self, n_nodes: int, cfg: Config, A_rel: float):
        self.n = int(n_nodes)
        self.cfg = cfg
        self.A_rel = float(A_rel)
        self.flow_A_per_year = float(cfg.flow_A_base_per_year) * self.A_rel
        self.L = float(cfg.domain_m)
        self.dx = self.L / (self.n - 1)
        self.ni = self.n - 2

        if self.ni <= 0:
            raise ValueError("Need at least 3 grid nodes.")
        if self.A_rel <= 0.0:
            raise ValueError("A_rel must be positive.")

        self.Nint = self.ni * self.ni
        self.last_u = np.zeros((2, self.n, self.n), dtype=np.float64)

    def reset_velocity(self) -> None:
        self.last_u.fill(0.0)

    def _effective_viscosity(self, vel: np.ndarray) -> np.ndarray:
        u, v = vel[0], vel[1]
        ux, uy = grad2(u, self.dx)
        vx, vy = grad2(v, self.dx)

        De2 = (
            ux * ux
            + vy * vy
            + ux * vy
            + 0.25 * (uy + vx) ** 2
            + self.cfg.strain_reg_per_year ** 2
        )
        De = np.sqrt(
            np.maximum(De2, self.cfg.strain_reg_per_year ** 2)
        )

        n = float(self.cfg.glen_n)
        A = float(self.flow_A_per_year)
        mu = 0.5 * A ** (-1.0 / n) * De ** (1.0 / n - 1.0)

        return np.clip(
            mu,
            self.cfg.viscosity_min_Pa_yr,
            self.cfg.viscosity_max_Pa_yr,
        )

    def _effective_beta(self, beta: np.ndarray) -> np.ndarray:
        p = float(self.cfg.friction_response_power)
        if abs(p - 1.0) > 1.0e-12:
            raise ValueError(
                "This controlled generator requires friction_response_power=1."
            )
        return np.asarray(beta, dtype=np.float64)

    def _idx(self, iy: int, ix: int) -> int:
        return (iy - 1) * self.ni + (ix - 1)

    def _build_matrix(self, K: np.ndarray, beta: np.ndarray) -> csc_matrix:
        rows = []
        cols = []
        vals = []
        h2 = self.dx * self.dx

        for iy in range(1, self.n - 1):
            for ix in range(1, self.n - 1):
                p = self._idx(iy, ix)

                ke = 0.5 * (K[iy, ix] + K[iy, ix + 1])
                kw = 0.5 * (K[iy, ix] + K[iy, ix - 1])
                kn = 0.5 * (K[iy, ix] + K[iy + 1, ix])
                ks = 0.5 * (K[iy, ix] + K[iy - 1, ix])

                diag = beta[iy, ix] + (ke + kw + kn + ks) / h2
                rows.append(p)
                cols.append(p)
                vals.append(diag)

                if ix + 1 <= self.n - 2:
                    rows.append(p)
                    cols.append(self._idx(iy, ix + 1))
                    vals.append(-ke / h2)
                if ix - 1 >= 1:
                    rows.append(p)
                    cols.append(self._idx(iy, ix - 1))
                    vals.append(-kw / h2)
                if iy + 1 <= self.n - 2:
                    rows.append(p)
                    cols.append(self._idx(iy + 1, ix))
                    vals.append(-kn / h2)
                if iy - 1 >= 1:
                    rows.append(p)
                    cols.append(self._idx(iy - 1, ix))
                    vals.append(-ks / h2)

        return coo_matrix(
            (vals, (rows, cols)),
            shape=(self.Nint, self.Nint),
        ).tocsc()

    def _rhs(self, H: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        Hx, Hy = grad2(H, self.dx)
        drive = self.cfg.rho_ice * self.cfg.gravity * H
        fx = -drive * Hx
        fy = -drive * Hy
        return (
            fx[1:-1, 1:-1].ravel(),
            fy[1:-1, 1:-1].ravel(),
        )

    def solve_velocity(
        self,
        H: np.ndarray,
        beta: np.ndarray,
    ) -> Tuple[np.ndarray, bool, int, float]:
        H = np.asarray(H, dtype=np.float64)
        beta = np.asarray(beta, dtype=np.float64)

        if H.shape != (self.n, self.n):
            raise ValueError("H grid shape mismatch.")
        if beta.shape != (self.n, self.n):
            raise ValueError("beta grid shape mismatch.")

        beta_eff = self._effective_beta(beta)
        vel = self.last_u.copy()

        if not np.any(vel):
            Hx, Hy = grad2(H, self.dx)
            denom = np.maximum(beta_eff, 1.0)
            vel[0] = -self.cfg.rho_ice * self.cfg.gravity * H * Hx / denom
            vel[1] = -self.cfg.rho_ice * self.cfg.gravity * H * Hy / denom
            vel[:, 0, :] = 0.0
            vel[:, -1, :] = 0.0
            vel[:, :, 0] = 0.0
            vel[:, :, -1] = 0.0

        rhs_u, rhs_v = self._rhs(H)
        final_rel = np.inf
        converged = False

        for nit in range(1, self.cfg.max_picard_iters + 1):
            mu = self._effective_viscosity(vel)
            Hstress = np.maximum(H, self.cfg.min_H_for_stress_m)
            K = self.cfg.membrane_scale * 2.0 * mu * Hstress

            M = self._build_matrix(K, beta_eff)
            lu = splu(M)
            uint = lu.solve(rhs_u)
            vint = lu.solve(rhs_v)

            new = np.zeros_like(vel)
            new[0, 1:-1, 1:-1] = uint.reshape(self.ni, self.ni)
            new[1, 1:-1, 1:-1] = vint.reshape(self.ni, self.ni)

            mask = H <= 1.0e-6
            new[:, mask] = 0.0

            final_rel = (
                np.linalg.norm((new - vel).ravel())
                / (np.linalg.norm(new.ravel()) + 1.0e-12)
            )

            if final_rel < self.cfg.picard_tol:
                vel = new
                converged = True
                break

            a = self.cfg.picard_relax
            vel = a * new + (1.0 - a) * vel

        self.last_u = vel.copy()
        return vel, converged, nit, float(final_rel)


# =============================================================================
# THICKNESS TRANSPORT
# =============================================================================


def upwind_transport_step(
    H: np.ndarray,
    u: np.ndarray,
    v: np.ndarray,
    dt: float,
    dx_m: float,
) -> np.ndarray:
    """Conservative update of dH/dt + div(H ubar) = 0."""
    ny, nx = H.shape

    ux = 0.5 * (u[:, :-1] + u[:, 1:])
    Hx_up = np.where(ux >= 0.0, H[:, :-1], H[:, 1:])
    Fx_int = ux * Hx_up
    Fx = np.zeros((ny, nx + 1), dtype=np.float64)
    Fx[:, 1:nx] = Fx_int

    vy = 0.5 * (v[:-1, :] + v[1:, :])
    Hy_up = np.where(vy >= 0.0, H[:-1, :], H[1:, :])
    Fy_int = vy * Hy_up
    Fy = np.zeros((ny + 1, nx), dtype=np.float64)
    Fy[1:ny, :] = Fy_int

    div = (
        (Fx[:, 1:] - Fx[:, :-1]) / dx_m
        + (Fy[1:, :] - Fy[:-1, :]) / dx_m
    )

    Hn = H - dt * div
    Hn[0, :] = 0.0
    Hn[-1, :] = 0.0
    Hn[:, 0] = 0.0
    Hn[:, -1] = 0.0
    return Hn


def advance_thickness(
    H: np.ndarray,
    vel: np.ndarray,
    years: float,
    dx_m: float,
    cfg: Config,
) -> np.ndarray:
    u, v = vel[0], vel[1]

    vmax = max(
        float(np.max(np.abs(u))),
        float(np.max(np.abs(v))),
        1.0e-12,
    )

    dt_cfl = cfg.cfl * dx_m / vmax
    dt = min(float(years), cfg.max_transport_substep_yr, dt_cfl)
    nsub = max(1, int(math.ceil(years / max(dt, 1.0e-12))))
    dt = years / nsub

    Hn = H.copy()
    for _ in range(nsub):
        Hn = upwind_transport_step(Hn, u, v, dt, dx_m)
        Hn = np.clip(Hn, 0.0, cfg.thickness_clip_max_m)
    return Hn




# =============================================================================
# ROBUST PICARD + T=50 TRAJECTORY SIMULATION
# =============================================================================


def solve_velocity_robust(
    solver: ReducedIceFlowSolver,
    H: np.ndarray,
    beta_grid: np.ndarray,
    cfg: Config,
    context: str,
) -> Tuple[np.ndarray, bool, int, float, int, float]:
    """Adaptive under-relaxation; equations and tolerance remain unchanged."""
    start_vel = solver.last_u.copy()
    original_relax = float(cfg.picard_relax)

    try:
        for alpha in PICARD_RELAX_CANDIDATES:
            solver.last_u = start_vel.copy()
            cfg.picard_relax = float(alpha)

            total_iters = 0
            rel = np.inf
            vel = None

            for block in range(1, MAX_PICARD_BLOCKS + 1):
                vel, conv, nit, rel = solver.solve_velocity(H, beta_grid)
                total_iters += int(nit)

                if conv:
                    return (
                        vel,
                        True,
                        total_iters,
                        float(rel),
                        block,
                        float(alpha),
                    )

        solver.last_u = start_vel.copy()
        raise RuntimeError(
            "Adaptive Picard failed for "
            f"{context}. Tried relaxations={PICARD_RELAX_CANDIDATES}, "
            f"{MAX_PICARD_BLOCKS} blocks x {cfg.max_picard_iters} iterations."
        )
    finally:
        cfg.picard_relax = original_relax





# =============================================================================
# TRAJECTORY GENERATION: ONE LF + ONE HF TRAJECTORY PER OPERATOR
# =============================================================================

def trajectory_cache_path(
    cache_dir: Path,
    operator_idx: int,
    fidelity: str,
) -> Path:
    op_dir = cache_dir / f"op_{operator_idx:03d}"
    op_dir.mkdir(parents=True, exist_ok=True)
    return op_dir / f"{fidelity}_trajectory_t1to50.npz"


def simulate_trajectory(
    solver: ReducedIceFlowSolver,
    beta_grid: np.ndarray,
    cfg: Config,
    operator_idx: int,
    fidelity: str,
    cache_path: Path,
) -> Dict[str, np.ndarray]:
    """
    Evolve one physical system from t=0 to t=50 and save COMPLETE snapshots
    at t=1,2,...,50.
    """
    if cache_path.exists() and not OVERWRITE_RUNS:
        z = np.load(cache_path, allow_pickle=False)
        required = {
            "time_values", "H", "U", "A_rel", "A_value",
            "operator_idx", "fidelity",
            "yearly_picard_rel", "yearly_picard_blocks",
            "yearly_picard_relax",
        }
        if required.issubset(set(z.files)):
            if (
                z["H"].shape[0] == cfg.n_saved_times
                and z["U"].shape[0] == cfg.n_saved_times
            ):
                print(f"[cache] op={operator_idx:03d} {fidelity}: {cache_path}")
                return {k: z[k] for k in z.files}

    n = solver.n
    _, _, X, Y = grid_xy(n, cfg.domain_m)
    H = halfar_initial_thickness(X, Y, cfg)
    solver.reset_velocity()

    time_values = np.arange(
        1, cfg.final_year + 1, dtype=np.float32
    )
    if len(time_values) != cfg.n_saved_times:
        raise ValueError(
            "Canonical dataset expects exactly one saved snapshot per year "
            "for t=1..50."
        )

    H_hist = np.empty(
        (cfg.n_saved_times, n, n), dtype=np.float32
    )
    U_hist = np.empty(
        (cfg.n_saved_times, 2, n, n), dtype=np.float32
    )
    yearly_rel = np.empty(cfg.n_saved_times, dtype=np.float32)
    yearly_blocks = np.empty(cfg.n_saved_times, dtype=np.int16)
    yearly_relax = np.empty(cfg.n_saved_times, dtype=np.float32)
    yearly_iters = np.empty(cfg.n_saved_times, dtype=np.int32)

    context_prefix = (
        f"op={operator_idx:03d} Arel={solver.A_rel:.6f} "
        f"fidelity={fidelity}"
    )

    # Initial t=0 velocity is only used to advance thickness to year 1.
    vel, _, nit0, rel0, blocks0, relax0 = solve_velocity_robust(
        solver,
        H,
        beta_grid,
        cfg,
        context=f"{context_prefix} t=0",
    )

    worst_rel = float(rel0)
    max_blocks = int(blocks0)
    min_relax = float(relax0)
    total_picard_iters = int(nit0)

    for k in range(cfg.n_saved_times):
        t = float(k + 1)

        H = advance_thickness(
            H,
            vel,
            cfg.evolution_step_years,
            solver.dx,
            cfg,
        )

        vel, _, nit, rel, blocks, relax_used = solve_velocity_robust(
            solver,
            H,
            beta_grid,
            cfg,
            context=f"{context_prefix} t={t:.1f}",
        )

        if not np.isfinite(H).all():
            raise FloatingPointError(
                f"Non-finite H at {context_prefix} t={t:.1f}"
            )
        if not np.isfinite(vel).all():
            raise FloatingPointError(
                f"Non-finite U at {context_prefix} t={t:.1f}"
            )

        H_hist[k] = H.astype(np.float32)
        U_hist[k] = vel.astype(np.float32)
        yearly_rel[k] = float(rel)
        yearly_blocks[k] = int(blocks)
        yearly_relax[k] = float(relax_used)
        yearly_iters[k] = int(nit)

        worst_rel = max(worst_rel, float(rel))
        max_blocks = max(max_blocks, int(blocks))
        min_relax = min(min_relax, float(relax_used))
        total_picard_iters += int(nit)

    payload = dict(
        time_values=time_values,
        H=H_hist,
        U=U_hist,
        A_rel=np.array(solver.A_rel, dtype=np.float64),
        A_value=np.array(solver.flow_A_per_year, dtype=np.float64),
        operator_idx=np.array(operator_idx, dtype=np.int64),
        fidelity=np.array(fidelity),
        yearly_picard_rel=yearly_rel,
        yearly_picard_blocks=yearly_blocks,
        yearly_picard_relax=yearly_relax,
        yearly_picard_iters=yearly_iters,
        t0_picard_rel=np.array(rel0, dtype=np.float64),
        t0_picard_blocks=np.array(blocks0, dtype=np.int16),
        t0_picard_relax=np.array(relax0, dtype=np.float32),
        worst_picard_rel=np.array(worst_rel, dtype=np.float64),
        max_picard_blocks=np.array(max_blocks, dtype=np.int16),
        min_picard_relax=np.array(min_relax, dtype=np.float32),
        total_picard_iters=np.array(total_picard_iters, dtype=np.int32),
    )
    np.savez_compressed(cache_path, **payload)

    speed50 = np.sqrt(
        (U_hist[-1].astype(np.float64) ** 2).sum(axis=0)
    )
    print(
        f"[done] op={operator_idx:03d} {fidelity} "
        f"Arel={solver.A_rel:.6f} | "
        f"H50max={float(H_hist[-1].max()):.2f} m | "
        f"U50max={float(speed50.max()):.3f} m/yr | "
        f"worstRel={worst_rel:.2e} | "
        f"maxBlocks={max_blocks} | minRelax={min_relax:.2f}"
    )
    return payload


def resolve_operator_slice(cfg: Config) -> Tuple[int, int]:
    start = int(OP_START_ENV) if OP_START_ENV else 0
    stop = int(OP_STOP_ENV) if OP_STOP_ENV else cfg.n_operators
    if not (0 <= start <= stop <= cfg.n_operators):
        raise ValueError(
            f"Invalid operator slice [{start}, {stop}) "
            f"for n_operators={cfg.n_operators}."
        )
    return start, stop


def generate_operator_range(
    cfg: Config,
    operator_bank: Dict[str, np.ndarray],
    out_dir: Path,
) -> None:
    cache = out_dir / "runs"
    cache.mkdir(parents=True, exist_ok=True)

    A_rel_values = operator_bank["A_rel_values"].astype(np.float64)
    beta_lf = operator_bank["beta_lf"].astype(np.float64)
    beta_hf = operator_bank["beta_hf"].astype(np.float64)

    op_start, op_stop = resolve_operator_slice(cfg)
    print(f"\n[generation] operator slice = [{op_start}, {op_stop})")

    for op_idx in range(op_start, op_stop):
        A_rel = float(A_rel_values[op_idx])

        print("\n" + "=" * 116)
        print(
            f"OPERATOR {op_idx:03d}/{cfg.n_operators - 1:03d} | "
            f"system=(A_rel={A_rel:.8f}, beta_{op_idx:03d}) | "
            f"A={cfg.flow_A_base_per_year * A_rel:.6e}"
        )
        print("=" * 116)

        lf_solver = ReducedIceFlowSolver(
            cfg.n_lf, cfg, A_rel=A_rel
        )
        hf_solver = ReducedIceFlowSolver(
            cfg.n_hf, cfg, A_rel=A_rel
        )

        lf_path = trajectory_cache_path(
            cache, op_idx, "LF"
        )
        hf_path = trajectory_cache_path(
            cache, op_idx, "HF"
        )

        if OVERWRITE_RUNS or not lf_path.exists():
            simulate_trajectory(
                lf_solver,
                beta_lf[op_idx],
                cfg,
                operator_idx=op_idx,
                fidelity="LF",
                cache_path=lf_path,
            )
        else:
            print(f"[exists] {lf_path}")

        if OVERWRITE_RUNS or not hf_path.exists():
            simulate_trajectory(
                hf_solver,
                beta_hf[op_idx],
                cfg,
                operator_idx=op_idx,
                fidelity="HF",
                cache_path=hf_path,
            )
        else:
            print(f"[exists] {hf_path}")



# =============================================================================
# MASTER ASSEMBLY
# =============================================================================

def load_cached_trajectory(
    cache: Path,
    operator_idx: int,
    fidelity: str,
) -> Dict[str, np.ndarray]:
    path = trajectory_cache_path(cache, operator_idx, fidelity)
    if not path.exists():
        raise FileNotFoundError(path)
    z = np.load(path, allow_pickle=False)
    return {k: z[k] for k in z.files}


def check_all_caches_exist(cfg: Config, out_dir: Path) -> None:
    cache = out_dir / "runs"
    missing = []

    for op_idx in range(cfg.n_operators):
        for fidelity in ("LF", "HF"):
            p = trajectory_cache_path(cache, op_idx, fidelity)
            if not p.exists():
                missing.append(str(p))
                if len(missing) >= 10:
                    break
        if len(missing) >= 10:
            break

    if missing:
        raise FileNotFoundError(
            "Cannot assemble master: trajectory caches are missing. "
            "First examples:\n" + "\n".join(missing)
        )


def sha256_file(path: Path, chunk_size: int = 8 * 1024 * 1024) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


def assemble_master(
    cfg: Config,
    operator_bank: Dict[str, np.ndarray],
    out_dir: Path,
) -> Path:
    check_all_caches_exist(cfg, out_dir)
    cache = out_dir / "runs"

    nO = cfg.n_operators
    nT = cfg.n_saved_times

    H_lf = np.empty(
        (nO, nT, cfg.n_lf, cfg.n_lf),
        dtype=np.float32,
    )
    U_lf = np.empty(
        (nO, nT, 2, cfg.n_lf, cfg.n_lf),
        dtype=np.float32,
    )
    H_hf = np.empty(
        (nO, nT, cfg.n_hf, cfg.n_hf),
        dtype=np.float32,
    )
    U_hf = np.empty(
        (nO, nT, 2, cfg.n_hf, cfg.n_hf),
        dtype=np.float32,
    )

    yearly_rel_lf = np.empty((nO, nT), dtype=np.float32)
    yearly_rel_hf = np.empty((nO, nT), dtype=np.float32)
    yearly_blocks_lf = np.empty((nO, nT), dtype=np.int16)
    yearly_blocks_hf = np.empty((nO, nT), dtype=np.int16)
    yearly_relax_lf = np.empty((nO, nT), dtype=np.float32)
    yearly_relax_hf = np.empty((nO, nT), dtype=np.float32)

    worst_rel_lf = np.empty(nO, dtype=np.float32)
    worst_rel_hf = np.empty(nO, dtype=np.float32)
    max_blocks_lf = np.empty(nO, dtype=np.int16)
    max_blocks_hf = np.empty(nO, dtype=np.int16)
    min_relax_lf = np.empty(nO, dtype=np.float32)
    min_relax_hf = np.empty(nO, dtype=np.float32)

    expected_times = np.arange(
        1, cfg.final_year + 1, dtype=np.float32
    )

    print("\n[assembly] loading full t=1..50 trajectories ...")
    for op_idx in range(nO):
        dl = load_cached_trajectory(
            cache, op_idx, "LF"
        )
        dh = load_cached_trajectory(
            cache, op_idx, "HF"
        )

        if not np.array_equal(dl["time_values"], expected_times):
            raise RuntimeError(
                f"LF time axis mismatch at operator {op_idx}."
            )
        if not np.array_equal(dh["time_values"], expected_times):
            raise RuntimeError(
                f"HF time axis mismatch at operator {op_idx}."
            )

        H_lf[op_idx] = dl["H"]
        U_lf[op_idx] = dl["U"]
        H_hf[op_idx] = dh["H"]
        U_hf[op_idx] = dh["U"]

        yearly_rel_lf[op_idx] = dl["yearly_picard_rel"]
        yearly_rel_hf[op_idx] = dh["yearly_picard_rel"]
        yearly_blocks_lf[op_idx] = dl["yearly_picard_blocks"]
        yearly_blocks_hf[op_idx] = dh["yearly_picard_blocks"]
        yearly_relax_lf[op_idx] = dl["yearly_picard_relax"]
        yearly_relax_hf[op_idx] = dh["yearly_picard_relax"]

        worst_rel_lf[op_idx] = float(dl["worst_picard_rel"])
        worst_rel_hf[op_idx] = float(dh["worst_picard_rel"])
        max_blocks_lf[op_idx] = int(dl["max_picard_blocks"])
        max_blocks_hf[op_idx] = int(dh["max_picard_blocks"])
        min_relax_lf[op_idx] = float(dl["min_picard_relax"])
        min_relax_hf[op_idx] = float(dh["min_picard_relax"])

        if (op_idx + 1) % 10 == 0 or op_idx == 0:
            print(f"  loaded operators {op_idx + 1}/{nO}")

    train_op = operator_bank[
        "train_operator_indices"
    ].astype(np.int64)
    test_op = operator_bank[
        "test_operator_indices"
    ].astype(np.int64)

    time_values = expected_times

    metadata = asdict(cfg)
    metadata.update(
        {
            "version": "ICE_LEVEL2_Abeta100_time1to50_fullfield_noFixedTimeSplit_v2",
            "operator_definition": (
                "operator_j = (A_j, beta_j(x,y)); both are fixed "
                "for the full t=0..50 trajectory"
            ),
            "A_definition": (
                "A_j = flow_A_base_per_year * A_rel_j"
            ),
            "operator_range_Arel": [
                cfg.A_rel_min,
                cfg.A_rel_max,
            ],
            "operator_beta_design": (
                "one independent GP beta realization per operator; "
                "beta is NOT shared across operators"
            ),
            "operator_split": (
                "fixed random 70 train / 30 completely held-out ID "
                "physical systems within A_rel interval"
            ),
            "within_operator_axis": (
                "time snapshots t=1,2,...,50 from one continuous trajectory"
            ),
            "lf_hf_relation": (
                "same physical (A_j,beta_j) at both fidelities; "
                "only spatial grid resolution differs"
            ),
            "saved_field_mode": (
                "complete H/U fields at every integer year 1..50"
            ),
            "model_hidden_metadata": ["A", "A_rel", "beta", "H"],
            "intended_icl_demo_rows": ["XY", "time", "Uc", "Uf"],
            "intended_icl_query_rows": ["XY", "time", "Uc"],
            "intended_target": "direct full Uf at query time",
            "intended_train_K": "dynamic 1..5, same train operator",
            "episode_partition_in_dataset": False,
            "time_sampling_policy": (
                "no demo/query times are hard-coded in the dataset; "
                "training/evaluation code samples distinct times from the "
                "same operator trajectory"
            ),
            "evaluation_reproducibility": (
                "use a fixed RNG seed in evaluation code if a reproducible "
                "support/query partition is desired"
            ),
            "ood_protocol": (
                "none; held-out systems are an in-distribution operator/system "
                "split and all saved times t=1..50 remain available"
            ),
            "Uc_source": (
                "LF velocity 15x15; intended loader upsamples to 41x41"
            ),
            "Uf_source": "HF velocity 41x41",
            "H_model_input": False,
            "beta_model_input": False,
            "A_model_input": False,
            "time_model_input": True,
            "solver_protocol": (
                "same reduced membrane-style equations at LF/HF; "
                "adaptive Picard under-relaxation alpha in "
                "{0.65,0.50,0.35,0.25}; yearly solves must converge"
            ),
        }
    )

    master_path = out_dir / cfg.master_name
    print(f"[assembly] writing compressed master: {master_path}")

    np.savez_compressed(
        master_path,
        # Grids / time.
        x_lf=operator_bank["x_lf"].astype(np.float32),
        y_lf=operator_bank["y_lf"].astype(np.float32),
        x_hf=operator_bank["x_hf"].astype(np.float32),
        y_hf=operator_bank["y_hf"].astype(np.float32),
        time_values=time_values.astype(np.float32),

        # Operator/system metadata.
        A_rel_values=operator_bank[
            "A_rel_values"
        ].astype(np.float32),
        A_values=operator_bank[
            "A_values"
        ].astype(np.float64),
        beta_lf=operator_bank[
            "beta_lf"
        ].astype(np.float32),
        beta_hf=operator_bank[
            "beta_hf"
        ].astype(np.float32),
        train_operator_indices=train_op,
        test_operator_indices=test_op,
        operator_split_labels=operator_bank[
            "operator_split_labels"
        ].astype(np.int8),

        # Full trajectory states.
        H_lf=H_lf,
        U_lf=U_lf,
        H_hf=H_hf,
        U_hf=U_hf,

        # Convergence audit.
        yearly_picard_rel_lf=yearly_rel_lf,
        yearly_picard_rel_hf=yearly_rel_hf,
        yearly_picard_blocks_lf=yearly_blocks_lf,
        yearly_picard_blocks_hf=yearly_blocks_hf,
        yearly_picard_relax_lf=yearly_relax_lf,
        yearly_picard_relax_hf=yearly_relax_hf,
        worst_picard_rel_lf=worst_rel_lf,
        worst_picard_rel_hf=worst_rel_hf,
        max_picard_blocks_lf=max_blocks_lf,
        max_picard_blocks_hf=max_blocks_hf,
        min_picard_relax_lf=min_relax_lf,
        min_picard_relax_hf=min_relax_hf,

        metadata_json=np.array(
            json.dumps(metadata, indent=2)
        ),
    )

    with open(out_dir / "metadata.json", "w") as f:
        json.dump(metadata, f, indent=2)

    digest = sha256_file(master_path)
    with open(out_dir / "master_sha256.txt", "w") as f:
        f.write(digest + "  " + master_path.name + "\n")

    print("[assembly] SHA256:", digest)
    return master_path


# =============================================================================
# SANITY REPORT
# =============================================================================

def sanity_report(path: Path, cfg: Config) -> None:
    z = np.load(path, allow_pickle=False)

    print("\n" + "=" * 124)
    print("ICE-SHEET LEVEL-2 TRAJECTORY ICL DATASET SANITY REPORT")
    print("=" * 124)
    print("Master                         :", path)
    print("Operators                      :", len(z["A_rel_values"]))
    print(
        "Train/test operators           :",
        len(z["train_operator_indices"]),
        "/",
        len(z["test_operator_indices"]),
    )
    print("time_values                    :", z["time_values"])
    print("beta_lf                        :", z["beta_lf"].shape)
    print("beta_hf                        :", z["beta_hf"].shape)
    print("H_lf                           :", z["H_lf"].shape)
    print("U_lf                           :", z["U_lf"].shape)
    print("H_hf                           :", z["H_hf"].shape)
    print("U_hf                           :", z["U_hf"].shape)
    expected = {
        "beta_lf": (
            cfg.n_operators, cfg.n_lf, cfg.n_lf
        ),
        "beta_hf": (
            cfg.n_operators, cfg.n_hf, cfg.n_hf
        ),
        "H_lf": (
            cfg.n_operators,
            cfg.n_saved_times,
            cfg.n_lf,
            cfg.n_lf,
        ),
        "U_lf": (
            cfg.n_operators,
            cfg.n_saved_times,
            2,
            cfg.n_lf,
            cfg.n_lf,
        ),
        "H_hf": (
            cfg.n_operators,
            cfg.n_saved_times,
            cfg.n_hf,
            cfg.n_hf,
        ),
        "U_hf": (
            cfg.n_operators,
            cfg.n_saved_times,
            2,
            cfg.n_hf,
            cfg.n_hf,
        ),
    }

    for key, shape in expected.items():
        if z[key].shape != shape:
            raise RuntimeError(
                f"Unexpected {key} shape: "
                f"{z[key].shape}, expected {shape}"
            )
        if not np.isfinite(z[key]).all():
            raise RuntimeError(
                f"Non-finite values found in {key}"
            )

    if not np.array_equal(
        z["time_values"],
        np.arange(1, 51, dtype=np.float32),
    ):
        raise RuntimeError(
            "Time axis is not exactly t=1..50."
        )

    train_set = set(
        map(int, z["train_operator_indices"].tolist())
    )
    test_set = set(
        map(int, z["test_operator_indices"].tolist())
    )
    if train_set & test_set:
        raise RuntimeError("Train/test operator overlap.")
    if len(train_set | test_set) != cfg.n_operators:
        raise RuntimeError(
            "Train/test split does not cover all operators."
        )

    worst_lf = float(z["worst_picard_rel_lf"].max())
    worst_hf = float(z["worst_picard_rel_hf"].max())
    print(
        f"Worst Picard rel LF/HF         : "
        f"{worst_lf:.3e} / {worst_hf:.3e}"
    )
    print(
        "Max Picard blocks LF/HF       :",
        int(z["max_picard_blocks_lf"].max()),
        "/",
        int(z["max_picard_blocks_hf"].max()),
    )
    print(
        "Min relaxation LF/HF          :",
        float(z["min_picard_relax_lf"].min()),
        "/",
        float(z["min_picard_relax_hf"].min()),
    )

    if worst_lf >= cfg.picard_tol or worst_hf >= cfg.picard_tol:
        raise RuntimeError(
            "Stored convergence audit violates picard_tol."
        )

    print("operator=(A,beta)              : PASS")
    print("one independent beta/operator  : PASS")
    print("one LF/HF trajectory/operator  : PASS")
    print("all t=1..50 snapshots saved    : PASS")
    print("operator-level 70/30 ID split  : PASS")
    print("predefined demo/query times     : NONE (by design)")
    print("model excludes A/beta/H        : metadata protocol PASS")
    print("all cached solves converged     : PASS")
    print("SHA256                         :", sha256_file(path))
    print("=" * 124)


# =============================================================================
# MAIN
# =============================================================================

def print_spec(
    cfg: Config,
    operator_bank: Dict[str, np.ndarray],
) -> None:
    print("=" * 124)
    print(
        "ICE-SHEET LEVEL-2 DATASET | "
        "100 (A,beta) OPERATORS x 50 TIME SNAPSHOTS"
    )
    print("=" * 124)
    print(
        "Operator definition             : "
        "operator_j = (A_j, beta_j(x,y))"
    )
    print(
        f"A_rel range                     : "
        f"[{cfg.A_rel_min}, {cfg.A_rel_max}]"
    )
    print(
        f"A base                          : "
        f"{cfg.flow_A_base_per_year:.6e} Pa^-3 yr^-1"
    )
    print(
        f"Operators                       : "
        f"{cfg.n_operators}"
    )
    print(
        f"Train/test operators            : "
        f"{cfg.n_train_operators} / "
        f"{cfg.n_operators - cfg.n_train_operators}"
    )
    print(
        "beta design                     : "
        "one independent GP field per operator"
    )
    print(
        "Within-operator axis            : "
        "t=1,2,...,50"
    )
    print(
        f"Snapshots/operator              : "
        f"{cfg.n_saved_times}"
    )
    print(
        f"Total paired snapshots          : "
        f"{cfg.n_operators * cfg.n_saved_times}"
    )
    print(
        f"LF/HF grids                     : "
        f"{cfg.n_lf}x{cfg.n_lf} / "
        f"{cfg.n_hf}x{cfg.n_hf}"
    )
    print(
        "Intended model rows             : "
        "[XY,t,Uc,Uf]xK + [XY,t,Uc] -> Uf_q"
    )
    print(
        "A / beta / H in model           : NO / NO / NO"
    )
    print(
        "Demo/query time partition       : "
        "NONE stored in dataset"
    )
    print(
        "Episode construction            : "
        "dynamic in train/eval code"
    )
    print(
        "OOD protocol                    : "
        "NONE; held-out systems are ID"
    )
    print("Output folder                   :", cfg.out_dir)
    print("Master                          :", cfg.master_name)
    print(
        "Train operator indices          :",
        operator_bank["train_operator_indices"].tolist(),
    )
    print(
        "Test operator indices           :",
        operator_bank["test_operator_indices"].tolist(),
    )
    print("=" * 124)


def main() -> None:
    cfg = build_config()

    # Lock the canonical scientific design.
    if cfg.n_operators != 100:
        raise ValueError("Canonical dataset requires n_operators=100.")
    if cfg.n_train_operators != 70:
        raise ValueError("Canonical dataset requires 70 train operators.")
    if cfg.n_saved_times != 50 or cfg.final_year != 50:
        raise ValueError(
            "Canonical dataset requires t=1..50."
        )
    if abs(cfg.evolution_step_years - 1.0) > 1.0e-12:
        raise ValueError(
            "Canonical dataset requires yearly evolution steps."
        )
    if abs(cfg.beta_gp_scale - 0.35) > 1.0e-12:
        raise ValueError(
            "Canonical dataset is locked to beta_gp_scale=0.35."
        )
    if abs(cfg.friction_response_power - 1.0) > 1.0e-12:
        raise ValueError(
            "Controlled dataset requires friction_response_power=1."
        )
    if abs(cfg.membrane_scale - 1.0) > 1.0e-12:
        raise ValueError(
            "Controlled dataset requires membrane_scale=1."
        )
    out_dir = Path(cfg.out_dir).resolve()
    out_dir.mkdir(parents=True, exist_ok=True)

    operator_bank = build_or_load_operator_bank(
        cfg, out_dir
    )
    print_spec(cfg, operator_bank)

    t0 = time.time()

    if not ASSEMBLE_ONLY:
        generate_operator_range(
            cfg,
            operator_bank,
            out_dir,
        )
    else:
        print(
            "[mode] ASSEMBLE_ONLY=1: "
            "skipping trajectory generation."
        )

    op_start, op_stop = resolve_operator_slice(cfg)
    full_generation_call = (
        op_start == 0 and op_stop == cfg.n_operators
    )

    if ASSEMBLE_ONLY or full_generation_call:
        master = assemble_master(
            cfg,
            operator_bank,
            out_dir,
        )
        sanity_report(master, cfg)
        print(f"\nSaved master: {master}")
    else:
        print(
            "\n[shard complete] Only a subset of operators "
            "was generated. After all shards exist, run:"
        )
        print("  ASSEMBLE_ONLY=1 python this_script.py")

    print(
        f"Elapsed wall time: "
        f"{(time.time() - t0) / 60.0:.2f} min"
    )


if __name__ == "__main__":
    main()


[beta] independent operator beta fields; GP scale s=0.35
[beta] Cholesky factorization of 1681 x 1681 covariance ...
[operators] Saved: /kaggle/working/ice_icl_Abeta100_time1to50_fullfield_noFixedTimeSplit/operator_bank_Abeta100_split70_30_gpScale0p35.npz
[operators] train indices: [0, 1, 2, 3, 5, 6, 7, 8, 10, 15, 16, 17, 18, 20, 23, 24, 25, 26, 27, 29, 30, 32, 33, 34, 35, 36, 37, 38, 39, 40, 42, 43, 44, 46, 47, 49, 51, 52, 53, 55, 56, 58, 59, 60, 61, 62, 63, 64, 66, 67, 68, 69, 70, 71, 75, 76, 78, 79, 81, 82, 83, 86, 87, 88, 90, 92, 94, 95, 96, 99]
[operators] test  indices: [4, 9, 11, 12, 13, 14, 19, 21, 22, 28, 31, 41, 45, 48, 50, 54, 57, 65, 72, 73, 74, 77, 80, 84, 85, 89, 91, 93, 97, 98]
ICE-SHEET LEVEL-2 DATASET | 100 (A,beta) OPERATORS x 50 TIME SNAPSHOTS
Operator definition             : operator_j = (A_j, beta_j(x,y))
A_rel range                     : [0.5, 2.0]
A base                          : 1.000000e-16 Pa^-3 yr^-1
Operators                       : 100
Train/test operator